In [19]:
import numpy as np
import pandas as pd
import warnings
import numpy as np

from itertools import combinations, product
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [20]:
df = pd.read_csv(r"../../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


## 전역

In [21]:
# =========================
# Global Best Subset Param Search Space
# =========================

param_space = {
    "C": [0.1, 0.5, 1.0, 3.0, 5.0, 10.0],
    "class_weight": ["balanced"],
    "solver": ["liblinear"],
    "threshold": np.arange(0.1, 0.91, 0.05)
}


def make_param_list(param_space):
    keys = list(param_space.keys())
    values = list(param_space.values())

    param_list = []

    for comb in product(*values):
        param_list.append(dict(zip(keys, comb)))

    return param_list


param_list = make_param_list(param_space)


# =========================
# Global Search Setting
# =========================
# 변수 28개 전체 전역탐색은 너무 큼.
# 그래서 단일 변수 성능 기준 top_k개만 남기고,
# 그 안에서 subset 조합을 전역탐색함.

global_search_config = {
    "top_k": 10,              # 28개 중 단일 성능 상위 10개만 후보로 사용
    "min_subset_size": 1,     # 최소 subset 크기
    "max_subset_size": 7,     # 최대 subset 크기
    "score_metric": "gmean",
    "random_state": 1,
    "max_iter": 2000,
    "verbose": True
}


print(f"총 파라미터 조합 수: {len(param_list)}")

top_k = global_search_config["top_k"]
max_subset_size = global_search_config["max_subset_size"]

estimated_subset_count = sum(
    len(list(combinations(range(top_k), r)))
    for r in range(
        global_search_config["min_subset_size"],
        max_subset_size + 1
    )
)

estimated_fit_count = estimated_subset_count * len(param_list)

print(f"후보 변수 수 top_k: {top_k}")
print(f"최대 subset 크기: {max_subset_size}")
print(f"예상 subset 조합 수: {estimated_subset_count}")
print(f"예상 모델 fit 횟수: {estimated_fit_count}")

총 파라미터 조합 수: 102
후보 변수 수 top_k: 10
최대 subset 크기: 7
예상 subset 조합 수: 967
예상 모델 fit 횟수: 98634


In [22]:
def get_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    gmean = np.sqrt(recall * spec)

    try:
        roc_auc = roc_auc_score(y_true, y_proba)
    except Exception:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true, y_proba)
    except Exception:
        pr_auc = np.nan

    metrics = {
        "gmean": gmean,
        "threshold": threshold,

        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision,
        "recall": recall,
        "spec": spec,
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),

        "roc_auc": roc_auc,
        "pr_auc": pr_auc,

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return metrics


def fit_eval_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    features,
    param_list,
    score_metric="gmean",
    random_state=1,
    max_iter=2000
):
    """
    특정 feature subset에 대해
    param_list 전체 탐색 후 validation score_metric이 가장 좋은 모델 반환.
    """

    best_record = None

    for params in param_list:
        model = LogisticRegression(
            penalty="l2",
            C=params["C"],
            class_weight=params["class_weight"],
            solver=params["solver"],
            max_iter=max_iter,
            random_state=random_state
        )

        model.fit(X_train[features], y_train)

        y_proba = model.predict_proba(X_valid[features])[:, 1]

        metrics = get_metrics(
            y_true=y_valid,
            y_proba=y_proba,
            threshold=params["threshold"]
        )

        record = {
            "features": list(features),
            "params": params,
            "metrics": metrics,
            "model": model
        }

        if best_record is None:
            best_record = record
        elif metrics[score_metric] > best_record["metrics"][score_metric]:
            best_record = record

    return best_record


def screen_features_by_single_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    param_list,
    top_k=10,
    score_metric="gmean",
    random_state=1,
    max_iter=2000,
    verbose=True
):
    """
    단일 변수 기준으로 먼저 후보 변수를 추림.
    28개 전체를 바로 전역탐색하면 너무 오래 걸리므로 사전 screening 용도.
    """

    records = []

    all_features = list(X_train.columns)

    if verbose:
        print("\n" + "=" * 80)
        print("1단계: 단일 변수 screening")
        print("=" * 80)
        print(f"전체 변수 수: {len(all_features)}")
        print(f"상위 {top_k}개 변수만 전역탐색 후보로 사용")

    for i, feature in enumerate(all_features, start=1):
        record = fit_eval_lr(
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            features=[feature],
            param_list=param_list,
            score_metric=score_metric,
            random_state=random_state,
            max_iter=max_iter
        )

        row = {
            "rank_temp": i,
            "feature": feature,
            "n_features": 1,
            **record["params"],
            **record["metrics"]
        }

        records.append(row)

        if verbose:
            print(
                f"[{i}/{len(all_features)}] {feature} | "
                f"{score_metric}={record['metrics'][score_metric]:.4f} | "
                f"threshold={record['params']['threshold']} | "
                f"C={record['params']['C']}"
            )

    screening_df = pd.DataFrame(records)

    screening_df = screening_df.sort_values(
        by=[score_metric, "f1", "recall", "spec", "acc"],
        ascending=False
    ).reset_index(drop=True)

    screening_df.insert(0, "rank", np.arange(1, len(screening_df) + 1))

    candidate_features = screening_df.head(top_k)["feature"].tolist()

    if verbose:
        print("\n선별된 후보 변수:")
        print(candidate_features)

    return candidate_features, screening_df


def global_best_subset_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    param_list,
    top_k=10,
    min_subset_size=1,
    max_subset_size=5,
    score_metric="gmean",
    random_state=1,
    max_iter=2000,
    verbose=True
):
    """
    현실형 전역선택법.

    1. 단일 변수 screening으로 top_k 후보 변수 선택
    2. 후보 변수 내에서 subset size min~max까지 모든 조합 평가
    3. 각 subset마다 param_list 전체 탐색
    4. validation score_metric 기준 best subset 선택

    주의:
        28개 전체 subset 전역탐색은 2^28 - 1개라 현실적으로 너무 오래 걸림.
    """

    X_train = X_train.copy()
    X_valid = X_valid.copy()

    y_train = np.asarray(y_train).astype(int)
    y_valid = np.asarray(y_valid).astype(int)

    if not isinstance(X_train, pd.DataFrame):
        raise TypeError("X_train은 DataFrame이어야 합니다. columns가 필요합니다.")

    if not isinstance(X_valid, pd.DataFrame):
        raise TypeError("X_valid는 DataFrame이어야 합니다. columns가 필요합니다.")

    if list(X_train.columns) != list(X_valid.columns):
        raise ValueError("X_train과 X_valid의 column 순서/이름이 같아야 합니다.")

    n_total_features = X_train.shape[1]

    if top_k is None:
        top_k = n_total_features

    top_k = min(top_k, n_total_features)

    if max_subset_size > top_k:
        max_subset_size = top_k

    if min_subset_size < 1:
        min_subset_size = 1

    if min_subset_size > max_subset_size:
        raise ValueError("min_subset_size는 max_subset_size보다 작거나 같아야 합니다.")

    candidate_features, screening_df = screen_features_by_single_lr(
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        param_list=param_list,
        top_k=top_k,
        score_metric=score_metric,
        random_state=random_state,
        max_iter=max_iter,
        verbose=verbose
    )

    subset_list = []

    for r in range(min_subset_size, max_subset_size + 1):
        for subset in combinations(candidate_features, r):
            subset_list.append(list(subset))

    total_subsets = len(subset_list)
    total_fit_count = total_subsets * len(param_list)

    if verbose:
        print("\n" + "=" * 80)
        print("2단계: 후보 변수 내 전역 subset 탐색")
        print("=" * 80)
        print(f"후보 변수 수: {len(candidate_features)}")
        print(f"subset size 범위: {min_subset_size} ~ {max_subset_size}")
        print(f"총 subset 조합 수: {total_subsets}")
        print(f"총 model fit 수: {total_fit_count}")

    best_record = None
    subset_records = []

    for idx, subset_features in enumerate(subset_list, start=1):
        record = fit_eval_lr(
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            features=subset_features,
            param_list=param_list,
            score_metric=score_metric,
            random_state=random_state,
            max_iter=max_iter
        )

        row = {
            "subset_id": idx,
            "n_features": len(subset_features),
            "features": subset_features,
            **record["params"],
            **record["metrics"]
        }

        subset_records.append(row)

        if best_record is None:
            best_record = record
        else:
            current_score = record["metrics"][score_metric]
            best_score = best_record["metrics"][score_metric]

            current_tie = (
                current_score,
                record["metrics"]["f1"],
                record["metrics"]["recall"],
                record["metrics"]["spec"],
                record["metrics"]["acc"],
                -len(record["features"])
            )

            best_tie = (
                best_score,
                best_record["metrics"]["f1"],
                best_record["metrics"]["recall"],
                best_record["metrics"]["spec"],
                best_record["metrics"]["acc"],
                -len(best_record["features"])
            )

            if current_tie > best_tie:
                best_record = record

        if verbose:
            if idx == 1 or idx % 50 == 0 or idx == total_subsets:
                print(
                    f"[{idx}/{total_subsets}] "
                    f"현재 best {score_metric}={best_record['metrics'][score_metric]:.4f} | "
                    f"best n_features={len(best_record['features'])}"
                )

    subset_history = pd.DataFrame(subset_records)

    subset_history = subset_history.sort_values(
        by=[score_metric, "f1", "recall", "spec", "acc", "n_features"],
        ascending=[False, False, False, False, False, True]
    ).reset_index(drop=True)

    result = {
        "selected_features": best_record["features"],
        "best_model": best_record["model"],
        "best_params": best_record["params"],
        "best_metrics": best_record["metrics"],

        "candidate_features": candidate_features,
        "screening_df": screening_df,
        "subset_history": subset_history,

        "n_total_features": n_total_features,
        "top_k": top_k,
        "min_subset_size": min_subset_size,
        "max_subset_size": max_subset_size,
        "total_subsets": total_subsets,
        "total_fit_count": total_fit_count,
        "score_metric": score_metric
    }

    return result

### 출력

In [23]:
global_result = global_best_subset_lr(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    param_list=param_list,

    top_k=global_search_config["top_k"],
    min_subset_size=global_search_config["min_subset_size"],
    max_subset_size=global_search_config["max_subset_size"],
    score_metric=global_search_config["score_metric"],

    random_state=global_search_config["random_state"],
    max_iter=global_search_config["max_iter"],
    verbose=global_search_config["verbose"]
)


metric_order = [
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "mcc",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]


print("\n" + "=" * 80)
print("전역선택 설정 요약")
print("=" * 80)
print(f"전체 변수 수: {global_result['n_total_features']}")
print(f"screening 후 후보 변수 수 top_k: {global_result['top_k']}")
print(f"subset size 범위: {global_result['min_subset_size']} ~ {global_result['max_subset_size']}")
print(f"실제 평가 subset 수: {global_result['total_subsets']}")
print(f"총 model fit 수: {global_result['total_fit_count']}")


print("\n" + "=" * 80)
print("Screening 후 후보 변수")
print("=" * 80)
print(global_result["candidate_features"])


print("\n" + "=" * 80)
print("최종 선택 변수")
print("=" * 80)
print(global_result["selected_features"])


print("\n" + "=" * 80)
print("Best Parameters")
print("=" * 80)
print(global_result["best_params"])


print("\n" + "=" * 80)
print("Validation Metrics")
print("=" * 80)

valid_metrics_df = pd.DataFrame([global_result["best_metrics"]])[metric_order]
display(valid_metrics_df)


print("\n" + "=" * 80)
print("Validation Confusion Matrix")
print("row=true, col=pred")
print("[[TN, FP],")
print(" [FN, TP]]")
print("=" * 80)

valid_cm = np.array([
    [global_result["best_metrics"]["tn"], global_result["best_metrics"]["fp"]],
    [global_result["best_metrics"]["fn"], global_result["best_metrics"]["tp"]]
])

print(valid_cm)


print("\n" + "=" * 80)
print("단일 변수 Screening Ranking Top 15")
print("=" * 80)

screening_cols = [
    "rank",
    "feature",
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "C",
    "class_weight",
    "solver"
]

display(global_result["screening_df"][screening_cols].head(15))


print("\n" + "=" * 80)
print("Global Subset Search Top 20")
print("=" * 80)

subset_cols = [
    "n_features",
    "features",
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "tn",
    "fp",
    "fn",
    "tp",
    "C",
    "class_weight",
    "solver"
]

display(global_result["subset_history"][subset_cols].head(20))


# =========================
# Test set 평가
# =========================

try:
    best_model = global_result["best_model"]
    selected_features = global_result["selected_features"]
    best_threshold = global_result["best_params"]["threshold"]

    y_test_proba = best_model.predict_proba(X_test[selected_features])[:, 1]

    test_metrics = get_metrics(
        y_true=y_test,
        y_proba=y_test_proba,
        threshold=best_threshold
    )

    print("\n" + "=" * 80)
    print("Test Metrics")
    print("=" * 80)

    test_metrics_df = pd.DataFrame([test_metrics])[metric_order]
    display(test_metrics_df)

    print("\n" + "=" * 80)
    print("Test Confusion Matrix")
    print("row=true, col=pred")
    print("[[TN, FP],")
    print(" [FN, TP]]")
    print("=" * 80)

    test_cm = np.array([
        [test_metrics["tn"], test_metrics["fp"]],
        [test_metrics["fn"], test_metrics["tp"]]
    ])

    print(test_cm)

except NameError:
    print("\nX_test 또는 y_test가 없어서 Test 평가는 생략됨.")


1단계: 단일 변수 screening
전체 변수 수: 28
상위 10개 변수만 전역탐색 후보로 사용


[1/28] KODEX 200_Premium | gmean=0.5127 | threshold=0.5000000000000001 | C=0.1
[2/28] KOSDAQ_return(%) | gmean=0.5427 | threshold=0.5000000000000001 | C=0.1
[3/28] NASDAQ_return(%) | gmean=0.6376 | threshold=0.5000000000000001 | C=0.1
[4/28] Brent Crude Oil_return(%) | gmean=0.5183 | threshold=0.5000000000000001 | C=0.1
[5/28] Gold Spot_return(%) | gmean=0.5701 | threshold=0.5000000000000001 | C=0.1
[6/28] USD/KRW_change(%) | gmean=0.5525 | threshold=0.5000000000000001 | C=0.1
[7/28] KOSPI 200 Volume | gmean=0.2730 | threshold=0.5500000000000002 | C=0.1
[8/28] return(%) | gmean=0.5374 | threshold=0.5000000000000001 | C=0.1
[9/28] KOSPI 200 lagged_1_return(%) | gmean=0.5197 | threshold=0.5000000000000001 | C=0.5
[10/28] KOSPI 200 lagged_2_return(%) | gmean=0.5519 | threshold=0.5000000000000001 | C=0.1
[11/28] VKOSPI_Close | gmean=0.5760 | threshold=0.5000000000000001 | C=0.1
[12/28] VKOSPI_Change(%) | gmean=0.4941 | threshold=0.5000000000000001 | C=0.1
[13/28] KOSPI 200_RSI14 | gmean=0.

,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.680499,0.45,0.659903,0.354391,0.23614,0.709877,0.652336,0.681106,0.250363,0.721668,0.346635,698,372,47,115



Validation Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[698 372]
 [ 47 115]]

단일 변수 Screening Ranking Top 15


,rank,feature,gmean,threshold,acc,f1,precision,recall,spec,C,class_weight,solver
0,1,NASDAQ_return(%),0.637607,0.5,0.708604,0.333952,0.238727,0.555556,0.731776,0.1,balanced,liblinear
1,2,VKOSPI_Close,0.576000,0.5,0.591721,0.263543,0.172745,0.555556,0.597196,0.1,balanced,liblinear
2,3,Gold Spot_return(%),0.570055,0.5,0.576299,0.258523,0.167897,0.561728,0.578505,0.1,balanced,liblinear
3,4,KOSPI 200_PPO_Hist,0.567372,0.5,0.562500,0.256552,0.165187,0.574074,0.560748,0.5,balanced,liblinear
4,5,GJR_VaR_5_t1,0.566323,0.5,0.648539,0.262351,0.181176,0.475309,0.674766,0.1,balanced,liblinear
5,6,KOSPI 200_RSI14,0.553704,0.5,0.499188,0.252121,0.156863,0.641975,0.477570,0.5,balanced,liblinear
6,7,KOSPI 200_BB_width,0.552818,0.5,0.596591,0.245827,0.162978,0.500000,0.611215,5.0,balanced,liblinear
7,8,USD/KRW_change(%),0.552546,0.5,0.533279,0.246396,0.156406,0.580247,0.526168,0.1,balanced,liblinear
8,9,KOSPI 200 lagged_2_return(%),0.551909,0.5,0.558442,0.244444,0.157706,0.543210,0.560748,0.1,balanced,liblinear
9,10,KOSPI 200_OG,0.550716,0.5,0.610390,0.245283,0.164557,0.481481,0.629907,0.1,balanced,liblinear



Global Subset Search Top 20


,n_features,features,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,tn,fp,fn,tp,C,class_weight,solver
0,5,"[NASDAQ_return(%), KOSPI 200_PPO_Hist, KOSPI 2...",0.680499,0.45,0.659903,0.354391,0.236140,0.709877,0.652336,0.681106,698,372,47,115,5.0,balanced,liblinear
1,6,"[NASDAQ_return(%), KOSPI 200_PPO_Hist, KOSPI 2...",0.680499,0.45,0.659903,0.354391,0.236140,0.709877,0.652336,0.681106,698,372,47,115,5.0,balanced,liblinear
2,6,"[NASDAQ_return(%), Gold Spot_return(%), KOSPI ...",0.680011,0.45,0.659091,0.353846,0.235656,0.709877,0.651402,0.680639,697,373,47,115,10.0,balanced,liblinear
3,7,"[NASDAQ_return(%), Gold Spot_return(%), KOSPI ...",0.680011,0.45,0.659091,0.353846,0.235656,0.709877,0.651402,0.680639,697,373,47,115,10.0,balanced,liblinear
4,6,"[NASDAQ_return(%), KOSPI 200_PPO_Hist, KOSPI 2...",0.679523,0.45,0.654221,0.352584,0.233871,0.716049,0.644860,0.680455,690,380,46,116,10.0,balanced,liblinear
5,7,"[NASDAQ_return(%), Gold Spot_return(%), KOSPI ...",0.679523,0.45,0.654221,0.352584,0.233871,0.716049,0.644860,0.680455,690,380,46,116,10.0,balanced,liblinear
6,7,"[NASDAQ_return(%), KOSPI 200_PPO_Hist, KOSPI 2...",0.679523,0.45,0.654221,0.352584,0.233871,0.716049,0.644860,0.680455,690,380,46,116,10.0,balanced,liblinear
7,4,"[NASDAQ_return(%), KOSPI 200_PPO_Hist, KOSPI 2...",0.675589,0.45,0.655844,0.349693,0.232653,0.703704,0.648598,0.676151,694,376,48,114,10.0,balanced,liblinear
8,5,"[NASDAQ_return(%), Gold Spot_return(%), KOSPI ...",0.675102,0.45,0.655032,0.349158,0.232179,0.703704,0.647664,0.675684,693,377,48,114,10.0,balanced,liblinear
9,4,"[NASDAQ_return(%), VKOSPI_Close, KOSPI 200_PPO...",0.675038,0.45,0.659091,0.349845,0.233471,0.697531,0.653271,0.675401,699,371,49,113,5.0,balanced,liblinear



Test Metrics


,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.67381,0.45,0.642336,0.309859,0.197605,0.717391,0.632877,0.675134,0.22484,0.736167,0.284662,462,268,26,66



Test Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[462 268]
 [ 26  66]]
